# CIM Datasets

The purpose of this notebook is to gather metadata for datasets found in [data.colorado.gov](data.colorado.gov), specifically those that have the Colorado Information Marketplace (CIM) listed as the dataset owner.

The SoDapy python package was used to retrieve the publicly available datasets, since it contains a `Socrata` object to use as a client for requests. This python package can be installed using

`pip install sodapy`

more information along with some examples can be found in
* [GitHub repo](https://github.com/xmunoz/sodapy#datasetslimit0-offset0) (archived)
* [API Docs](https://dev.socrata.com/consumers/getting-started.html)

Another link with a useful example for getting started was found [here](https://holowczak.com/getting-started-with-nyc-opendata-and-the-socrata-api/5/)

Required packages:


In [1]:
from sodapy import Socrata
import pandas as pd

Unfortunately most of the API calls that could be made with this client required a dataset id to be passed in as a paramater. Therefore all of the datasets found in the data catalog were pulled in and filtered using conditionals with pandas.

In [2]:
cim_url_query = 'data.colorado.gov'
datasets = None

with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()

Some discovery work showed that there wasn't an owner name found within each JSON, but there did exist an owner id. Since this notebook was intended to filter out any datasets not relating to CIM, it was safe to hard code the CIM name as the dataset owner. Another thing to note was that when the dataset JSON was converted to a pandas data frame, each row had a corresponding name instead of a numerical index. To locate the row of interest, the `.loc[[]]` syntax was used. After locating the necessary information found in the JSON, the row was appended to the result data frame.

In [ ]:
res = pd.DataFrame()
cim_dataset_owner_id = '8cet-tw9x'
for dset in datasets:
    dset_df = pd.DataFrame(dset)
    if (dset_df.loc[['type']]['resource'].item() == 'dataset'
        and dset_df.loc[['id']]['owner'].item() == cim_dataset_owner_id):
        page_views = dset_df.loc[['page_views']]['resource'].item()
        row = {
            "name": dset_df.loc[['name']]['resource'].item(),
            "id": dset_df.loc[['id']]['resource'].item(),
            "link": dset_df.loc[['id']]['link'].item(),
            "owner": 'Colorado Information Marketplace',
            "owner_id": dset_df.loc[['id']]['owner'].item(),
            "attribution": dset_df.loc[['attribution']]['resource'].item(),
            "attribution_link": dset_df.loc[['attribution_link']]['resource'].item(),
            "createdAt": dset_df.loc[['createdAt']]['resource'].item(),
            "data_updated_at": dset_df.loc[['data_updated_at']]['resource'].item(),
            "metadata_updated_at": dset_df.loc[['metadata_updated_at']]['resource'].item(),
            "publication_date": dset_df.loc[['publication_date']]['resource'].item(),
            "page_views_total": page_views['page_views_total'],
            "page_views_total_log": page_views['page_views_total_log'],
            "page_views_last_week": page_views['page_views_last_week'],
            "page_views_last_week_log": page_views['page_views_last_week_log'],
            "page_views_last_month": page_views['page_views_last_month'],
            "page_views_last_month_log": page_views['page_views_last_month_log'],
            "download_count": dset_df.loc[['download_count']]['resource'].item()
        }
        res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)

The results were written out to csv with index turned off to avoid a duplicate index column.

In [ ]:
res.head()

,name,id,link,owner,owner_id,attribution,attribution_link,createdAt,data_updated_at,metadata_updated_at,publication_date,page_views_total,page_views_total_log,page_views_last_week,page_views_last_week_log,page_views_last_month,page_views_last_month_log,download_count
0,Business Entities in Colorado,4ykn-tg5h,https://data.colorado.gov/Business/Business-En...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-03-19T22:33:57.000Z,2022-10-03T11:22:19.000Z,2022-10-03T11:01:14.000Z,2018-03-07T16:30:41.000Z,81926,16.322051,327,8.357552,1243,10.280771,23348
1,Professional and Occupational Licenses in Colo...,7s5z-vewr,https://data.colorado.gov/Regulations/Professi...,Colorado Information Marketplace,8cet-tw9x,DORA,https://www.colorado.gov/dora,2016-03-31T21:59:36.000Z,2022-10-03T09:31:56.000Z,2022-10-03T11:02:24.000Z,2018-06-07T19:20:49.000Z,19197,14.228668,51,5.700440,266,8.060696,2965
2,Population Projections in Colorado,q5vp-adf3,https://data.colorado.gov/Demographics/Populat...,Colorado Information Marketplace,8cet-tw9x,DOLA,https://www.colorado.gov/dola,2014-01-28T23:36:26.000Z,2021-11-01T10:31:11.000Z,2022-10-03T11:01:22.000Z,2021-01-12T22:35:05.000Z,18719,14.192293,79,6.321928,297,8.219169,2920
3,Trade Names for Businesses in Colorado,u7sb-g482,https://data.colorado.gov/Business/Trade-Names...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-01-08T19:34:11.000Z,2022-09-27T10:01:27.000Z,2022-10-03T11:02:55.000Z,2022-06-08T04:27:33.000Z,9612,13.230771,33,5.087463,147,7.209453,12037
4,Uniform Commercial Code (UCC) Filing Informati...,wffy-3uut,https://data.colorado.gov/Business/Uniform-Com...,Colorado Information Marketplace,8cet-tw9x,CDOS,https://www.sos.state.co.us/,2014-03-09T07:33:16.000Z,2022-10-03T09:05:29.000Z,2022-10-03T11:01:10.000Z,2016-11-22T23:54:53.000Z,9238,13.173521,59,5.906891,190,7.577429,2201


In [ ]:
res.to_csv('cim_datasets.csv', index=False)

In [3]:
datasets[0]

{'resource': {'name': 'Transparency Online Project (TOPS) - State Government Revenue and Expenditures in Colorado',
  'id': 'rifs-n6ib',
  'parent_fxf': [],
  'description': 'Revenue and Expenditures for Colorado State Governmental departments provided by the Department of Personnel & Administration and the Office of the State Controller. Documentation can be found in the long description of the dataset.',
  'attribution': 'DPA',
  'attribution_link': 'https://dpa.colorado.gov/',
  'contact_email': None,
  'type': 'story',
  'updatedAt': '2022-11-18T19:02:12.032Z',
  'createdAt': '2022-11-18T18:57:23.000Z',
  'metadata_updated_at': '2024-08-08T11:00:31.000Z',
  'data_updated_at': '2022-11-18T19:02:12.032Z',
  'page_views': {'page_views_last_week': 351,
   'page_views_last_month': 1403,
   'page_views_total': 548432,
   'page_views_last_week_log': 8.459431618637296,
   'page_views_last_month_log': 10.45532722030456,
   'page_views_total_log': 19.064955857195038},
  'columns_name': [],
 

In [5]:
datasets[10]

{'resource': {'name': 'Professional and Occupational Licenses in Colorado',
  'id': '7s5z-vewr',
  'parent_fxf': [],
  'description': 'Professional and occupational license types in Colorado from the Department of Regulatory Agencies (DORA).',
  'attribution': 'DORA',
  'attribution_link': 'https://www.colorado.gov/dora',
  'contact_email': None,
  'type': 'dataset',
  'updatedAt': '2024-08-08T11:01:35.000Z',
  'createdAt': '2016-03-31T21:59:36.000Z',
  'metadata_updated_at': '2024-08-08T11:01:35.000Z',
  'data_updated_at': '2024-08-08T09:42:39.000Z',
  'page_views': {'page_views_last_week': 516,
   'page_views_last_month': 1994,
   'page_views_total': 43286,
   'page_views_last_week_log': 9.014020470314934,
   'page_views_last_month_log': 10.962173031109709,
   'page_views_total_log': 15.401646197768974},
  'columns_name': ['licenseStatusDescription',
   'disciplineCompleteDate',
   'disciplineEffectiveDate',
   'programAction',
   'licenseExpirationDate',
   'licenseFirstIssueDate',


In [6]:
datasets[10]["owner"]["display_name"]

'Colorado Information Marketplace'

In [7]:
cim=[]
for dataset in datasets:
    if dataset["owner"]["display_name"] == 'Colorado Information Marketplace':
        cim.append(dataset)

In [8]:
len(cim)

602

In [9]:
cim[0]["resource"]["type"]

'map'

In [21]:
hist = {}
lic = {}
for dat in cim:
    tp = dat["resource"]["type"]
    if "license"  in dat["metadata"]:
       li = dat["metadata"]["license"]
       if li not in lic:
        lic[li]=0
       lic[li]+=1
    if tp not in hist:
        hist[tp]=0
    hist[tp]+=1

  

In [23]:
hist

{'map': 170, 'dataset': 412, 'filter': 6, 'href': 13, 'story': 1}

In [ ]:
cim

In [13]:
for col in cim[0]["resource"].keys():
    print(col)

name
id
parent_fxf
description
attribution
attribution_link
contact_email
type
updatedAt
createdAt
metadata_updated_at
data_updated_at
page_views
columns_name
columns_field_name
columns_datatype
columns_description
columns_format
download_count
provenance
lens_view_type
lens_display_type
locked
blob_mime_type
hide_from_data_json
publication_date


In [14]:
cim[0]

dict_keys(['resource', 'classification', 'metadata', 'permalink', 'link', 'preview_image_url', 'owner', 'creator'])

In [17]:
cim[0]["metadata"]

{'domain': 'data.colorado.gov', 'license': 'Public Domain'}